#Uploads - Bronze
1. Set up w/
    - Tables/hnm_articles_bronze (HuggingFace)
    - Tables/hnm_transactions_bronze (master transactions table)
2. Manual hnm_weekly_transactions uplaod
3. Hypothetically hnm_weekly_transactions Azure API call


In [0]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from datasets import load_dataset #%pip install datasets
import os

Articles setup through Hugging Face

In [0]:
# Articles from Hugging Face

# Volume (as parquet) 
volume_path = "/Volumes/workspace/default/kaggle_volume/hnm_raw"

dbutils.fs.mkdirs(volume_path)

csv_files = ["articles"]

for file_name in csv_files:
    # load
    hf_dataset = load_dataset("einrafh/hnm-fashion-recommendations-data", file_name)
    
    # extract 'train' split & save directly as Parquet
    parquet_path = f"{volume_path}/hnm_{file_name}.parquet"
    hf_dataset['train'].to_parquet(parquet_path)
    
    ## Parquet into PySpark & write to Delta Lake
    target_table = f"default.hnm_{file_name}_bronze"
    (spark.read.parquet(parquet_path)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table))

display(spark.table("workspace.default.hnm_articles_bronze"))

Upload master hnm_transaction_bronze file to Volumes from local drive since it is too large for the Databricks UI


In [0]:
# Not for Db, for personal drive upload, use in CLI 
pip install databricks-sdk

from databricks.sdk import WorkspaceClient

# Initializes using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

os.environ["DATABRICKS_HOST"] = "https://databricks.com"
os.environ["DATABRICKS_TOKEN"] = "your_token_here"

local_file_path = "/Users/File/Documents/hnm_transactions.csv"
volume_dest_path = "/Volumes/workspace/default/hnm_transactions/hnm_transactions.csv"

with open(local_file_path, "rb") as f:
    w.files.upload(volume_dest_path, f)
print("Upload complete.")


In [0]:

# Read files from volume
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/hnm_transactions/hnm_transactions.csv")

df.write.saveAsTable("workspace.default.hnm_transactions_bronze")


Manual weekly transaction upload into Volume (hnm_weekly_upload)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.hnm_weekly_upload")

Hypothetically Azure API call for weekly_transactions csv upload 

In [0]:
"""
- fetch weekly transactions week_(dates) for a hypothetical Azure API call 
- upload_csv() writes week_(dates).csv to the Unity Catalog volume
"""

from databricks.sdk import WorkspaceClient

w = WorkspaceClient(host="https://<workspace>.azuredatabricks.net", token="<token>")

with open("week_(dates).csv", "rb") as f:
    w.files.upload("/Volumes/workspace/default/hnm_weekly_upload/week_(dates).csv", f, overwrite=True)